# Data Quality Results Audit Logger

**Purpose:** Log data quality validation results for compliance evidence and operational monitoring.

**Target Table:** `workspace.audit.audit_dq_results`

**Event Type:** Data quality validation events

**Key Metrics:**
* Rule-level validation results
* Failed record counts and thresholds
* Severity classification (ERROR, WARNING, INFO)
* Target table/schema identification

**Usage:**
```python
dbutils.notebook.run(
    "LMIP/notebooks/audit/audit_dq_results",
    timeout_seconds=300,
    arguments={
        "run_id": "unique_run_identifier",
        "rule_name": "check_null_customer_id",
        "target_schema": "silver",
        "target_table": "customers",
        "failed_records": "5",
        "severity": "ERROR"
    }
)
```

In [0]:
# Notebook parameters - configured via dbutils.widgets or job parameters
dbutils.widgets.text("run_id", "", "Run ID (FK to pipeline runs)")
dbutils.widgets.text("rule_name", "", "DQ Rule Name")
dbutils.widgets.text("target_schema", "", "Target Schema")
dbutils.widgets.text("target_table", "", "Target Table")
dbutils.widgets.text("failed_records", "0", "Failed Records Count")
dbutils.widgets.text("severity", "INFO", "Severity (ERROR/WARNING/INFO)")
dbutils.widgets.text("rule_description", "", "Optional: Rule Description")

# Retrieve parameter values
run_id = dbutils.widgets.get("run_id")
rule_name = dbutils.widgets.get("rule_name")
target_schema = dbutils.widgets.get("target_schema")
target_table = dbutils.widgets.get("target_table")
failed_records = int(dbutils.widgets.get("failed_records") or 0)
severity = dbutils.widgets.get("severity")
rule_description = dbutils.widgets.get("rule_description")

print(f"DQ Audit: {rule_name} | Target: {target_schema}.{target_table} | Failures: {failed_records} | Severity: {severity}")

In [0]:
from pyspark.sql import functions as F
from datetime import datetime
import uuid

In [0]:
# Validation: Ensure mandatory parameters are provided
if not run_id:
    raise ValueError("Parameter 'run_id' is required")
    
if not rule_name:
    raise ValueError("Parameter 'rule_name' is required")
    
if not target_schema:
    raise ValueError("Parameter 'target_schema' is required")
    
if not target_table:
    raise ValueError("Parameter 'target_table' is required")
    
if severity not in ["ERROR", "WARNING", "INFO"]:
    raise ValueError(f"Invalid severity: {severity}. Must be ERROR, WARNING, or INFO")

print("✓ Parameter validation passed")

In [0]:
# Generate surrogate key
audit_dq_sk = int(str(uuid.uuid4().int)[:18])  # 18-digit bigint

# Capture evaluation timestamp
evaluated_at = datetime.now()

# Build DQ audit record
dq_audit_record = {
    "audit_dq_sk": audit_dq_sk,
    "run_id": run_id,
    "rule_name": rule_name,
    "target_schema": target_schema,
    "target_table": target_table,
    "failed_records": failed_records,
    "severity": severity,
    "evaluated_at": evaluated_at
}

print(f"Generated DQ audit record with SK: {audit_dq_sk}")
print(f"  - Rule: {rule_name}")
print(f"  - Target: {target_schema}.{target_table}")
print(f"  - Failed Records: {failed_records}")
print(f"  - Severity: {severity}")

In [0]:
# Create DataFrame from DQ audit record
dq_audit_df = spark.createDataFrame([dq_audit_record])

# Write to audit table (append mode)
target_table_name = "workspace.audit.audit_dq_results"

try:
    dq_audit_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(target_table_name)
    
    print(f"✓ Successfully logged DQ audit record to {target_table_name}")
    
    # Alert if critical failure
    if severity == "ERROR" and failed_records > 0:
        print(f"⚠️ CRITICAL: DQ rule '{rule_name}' failed with {failed_records} records")
        print(f"   Action required for {target_schema}.{target_table}")
    
except Exception as e:
    print(f"✗ Failed to write DQ audit record: {str(e)}")
    raise

In [0]:
%sql
-- Verify the DQ audit log was written successfully
-- Shows recent data quality validation results

SELECT 
    run_id,
    rule_name,
    CONCAT(target_schema, '.', target_table) as target_table,
    failed_records,
    severity,
    evaluated_at,
    CASE 
        WHEN severity = 'ERROR' AND failed_records > 0 THEN '⚠️ CRITICAL'
        WHEN severity = 'WARNING' AND failed_records > 0 THEN '⚠️ Warning'
        ELSE '✓ Pass'
    END as status_flag
FROM workspace.audit.audit_dq_results
ORDER BY evaluated_at DESC
LIMIT 10

In [0]:
%sql
-- Aggregate view of DQ results by severity
-- Useful for identifying systemic quality issues

SELECT 
    severity,
    COUNT(*) as total_checks,
    SUM(CASE WHEN failed_records > 0 THEN 1 ELSE 0 END) as failed_checks,
    SUM(failed_records) as total_failed_records,
    COUNT(DISTINCT CONCAT(target_schema, '.', target_table)) as affected_tables
FROM workspace.audit.audit_dq_results
WHERE evaluated_at >= CURRENT_DATE() - INTERVAL 7 DAYS
GROUP BY severity
ORDER BY 
    CASE severity 
        WHEN 'ERROR' THEN 1 
        WHEN 'WARNING' THEN 2 
        WHEN 'INFO' THEN 3 
    END

In [0]:
# Return success indicator with DQ status
result = {
    "status": "success",
    "audit_dq_sk": audit_dq_sk,
    "dq_passed": failed_records == 0,
    "severity": severity,
    "failed_records": failed_records
}

dbutils.notebook.exit(result)